
# Chapitre 1 : Exploration et Analyse de la Qualité des Données (EDA)

## 1.1 Introduction à la Phase d’Exploration
Cette première phase constitue le socle analytique du projet de détection du mélanome. Elle vise à auditer rigoureusement la structure du dataset HAM10000 (*Human Against Machine with 10000 training images*). L'objectif scientifique n'est pas de procéder à un simple dénombrement, mais de cartographier l'espace des caractéristiques (*feature space*) afin d'identifier les biais statistiques, les fuites de données potentielles et les artefacts visuels. Ces observations dicteront les choix architecturaux des phases ultérieures (prétraitement, modélisation par CNN et Random Forest).

## 1.2 Audit d'Intégrité et Complétude des Données
Un premier audit global a été réalisé sur les 10 015 instances constituant le corpus clinique. L'analyse de complétude a révélé une intégrité exceptionnelle des données tabulaires :
* **Taux de complétude :** Le dataset est renseigné à 99,43 %. Les variables critiques telles que l'identifiant de lésion, le type de diagnostic et la localisation corporelle sont totalement exemptes de valeurs nulles.
* **Gestion des valeurs manquantes :** Seule la variable continue « Âge » présente 57 occurrences non renseignées (soit 0,57 % du total).
* **Décision algorithmique :** Compte tenu du caractère statistiquement marginal de cette absence, une stratégie d'imputation par la médiane est privilégiée. Cette méthode permet de préserver la tendance centrale de la distribution démographique sans introduire de biais de variance, contrairement à une suppression de lignes qui entraînerait une perte (bien que minime) de données d'apprentissage visuel.

## 1.3 Analyse de la Distribution des Classes et Matrice des Coûts
L'examen de la variable cible (les sept types de lésions cutanées) a mis en exergue un défi majeur inhérent aux datasets médicaux : le déséquilibre extrême des classes (*Class Imbalance*).
* **Analyse quantitative :** La classe majoritaire, le Nævus mélanocytaire bénin (nv), domine massivement l'espace avec 67 % des occurrences (6 705 images). À l'inverse, le Mélanome (mel), pathologie critique et cible principale de notre système, ne représente que 11 % du corpus (1 113 images).

> **[Insérer l'image ici : `reports/figures/distribution_classes.png`]**
> *Figure 1.1 : Distribution asymétrique des 7 classes diagnostiques du dataset HAM10000.*

* **Implication clinique et algorithmique :** Dans un contexte d'aide au diagnostic médical, le coût d'un Faux Négatif (ne pas détecter un mélanome) est infiniment supérieur à celui d'un Faux Positif (classer un Nævus comme suspect). Ce ratio de déséquilibre impose l'implémentation future de techniques strictes : pondération pénalisante lors de l'apprentissage (*Class Weights*) et génération de données synthétiques par *Data Augmentation* pour la classe minoritaire.

## 1.4 Sécurité de l'Évaluation : Prévention de la Fuite de Données (*Data Leakage*)
L'analyse croisée du nombre d'images (10 015) et des identifiants uniques de lésions (`lesion_id`) a permis de lever un risque critique pour la validité du modèle. Le dataset ne contient en réalité que 7 470 lésions distinctes.
* **Constat :** Environ 25 % des lésions ont fait l'objet de multiples captures photographiques (variations de grossissement, d'angle ou d'éclairage lors de l'examen dermoscopique).
* **Risque d'Overfitting :** Une séparation aléatoire standard des données (de type *Train/Test Split*) exposerait le système à un biais de fuite de données (*Data Leakage*). Le réseau de neurones pourrait s'entraîner sur l'image A d'une lésion et être évalué sur l'image B de cette *même* lésion, évaluant ainsi sa capacité à mémoriser les caractéristiques visuelles d'un patient spécifique plutôt que les marqueurs réels de la pathologie.
* **Décision architecturale :** La validation croisée et la séparation des ensembles d'apprentissage et de test seront rigoureusement partitionnées par identifiant de lésion (via une méthode de type `GroupKFold`).

## 1.5 Analyse de la Fiabilité Diagnostique (Vérité Terrain)
La robustesse d'un modèle supervisé dépend intrinsèquement de la qualité de ses labels (la vérité terrain). L'analyse de la variable `dx_type` (méthode de confirmation du diagnostic) consolide la viabilité de ce projet. Plus de la moitié des diagnostics globaux, et la quasi-totalité des mélanomes, ont été confirmés par **histopathologie** (biopsie examinée au microscope). Cette méthode représente le standard de référence (Gold Standard) en médecine, assurant que l'algorithme s'entraînera sur une base de vérité quasi incontestable pour les cas de malignité.

## 1.6 Profilage Démographique et Localisation Corporelle
L'exploration des métadonnées cliniques a permis d'isoler des facteurs de risque quantifiables :

> **[Insérer l'image ici : `reports/figures/metadata_analysis.png`]**
> *Figure 1.2 : Répartition démographique (Âge, Sexe) et topographique des lésions cutanées.*

* **Dynamique de l'âge :** La distribution de l'âge est asymétrique, avec une prévalence maximale située dans la tranche des 45-60 ans, corroborant la littérature médicale liant l'apparition de lésions à l'accumulation d'expositions aux rayons UV.
* **Prédisposition anatomique :** L'analyse topographique isole la zone dorsale (*back*) comme la localisation prédominante. Plus critique encore, nos calculs révèlent que 29,1 % des mélanomes identifiés sont situés sur le dos, une zone hors du champ de vision naturel du patient, favorisant les diagnostics tardifs.

## 1.7 Analyse des Corrélations Multimodales
Afin de valider la pertinence d'une architecture multimodale (fusionnant images et données cliniques), une matrice de corrélation a été calculée en encodant bidairement les variables catégorielles.

> **[Insérer l'image ici : `reports/figures/correlation_matrix.png`]**
> *Figure 1.3 : Matrice de corrélation de Pearson entre les métadonnées et la malignité (Mélanome).*

Cette analyse mathématique confirme que des variables comme l'âge avancé et la localisation dorsale présentent une corrélation positive (signaux faibles mais cliniquement pertinents) avec le diagnostic de mélanome. Ces métadonnées constituent des descripteurs robustes qui alimenteront le modèle tabulaire (Random Forest).

## 1.8 Inspection Visuelle et Variabilité Intra/Inter-classes
La dernière étape de cette phase a consisté en l'extraction d'une grille d'échantillonnage représentative des 7 classes afin d'évaluer la complexité de la tâche de vision par ordinateur.

> **[Insérer l'image ici : `reports/figures/echantillons_classes.png`]**
> *Figure 1.4 : Grille d'échantillonnage aléatoire des lésions par classe diagnostique.*

L'inspection visuelle met en évidence deux défis majeurs pour le réseau de neurones convolutif (CNN) :
1. **Une faible variance inter-classe :** Les stades précoces du mélanome partagent de fortes similarités morphologiques (couleur, asymétrie naissante) avec certains Nævus atypiques.
2. **Une forte présence d'artefacts (bruit) :** De nombreuses images contiennent des éléments perturbateurs massifs (pilosité dense, bulles d'huile dermoscopique, marqueurs chirurgicaux au stylo). Ces observations justifient la nécessité absolue d'appliquer des filtres de prétraitement (ex: algorithme DullRazor pour l'élimination des poils) ou des transformations géométriques sévères lors de l'entraînement.

## 1.9 Conclusion de la Phase 1
L'exploration approfondie du dataset HAM10000 valide scientifiquement la faisabilité du projet tout en dictant les contraintes de l'ingénierie des données. La gestion du déséquilibre des classes, la sécurisation contre la fuite de données via le regroupement par lésion, et l'intégration des variables cliniques (Âge, Dos) constituent désormais les directives de conception pour la phase de prétraitement et modélisation qui fait l'objet du chapitre suivant.